# Network-Based Indicators vs TDA on the Same Distance Matrices

This notebook responds to the reviewer suggestion that the paper should compare TDA against simpler network-based indicators computed from the same distance matrices.

Design choices:

- We use the same 15-asset distance matrices that enter the VR/TDA pipeline.
- Because these matrices define dense weighted graphs, unweighted degree centrality and unweighted clustering coefficient would be degenerate or uninformative. We therefore compute weighted network counterparts.
- Distances are converted into affinity weights using the same method-specific global upper radius used for the VR filtration, so the network indicators and the TDA indicators are built from the same information source and scale.
- We then compare network indicators and TDA using the same multi-window AUC evaluation design.


In [1]:
from pathlib import Path
import json
import re
import time

import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_curve, auc
from IPython.display import display

SIM_ROOT = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_similarity_matrices')
VR_PARAM_PATH = Path('/Users/jane/Documents/202511吾-Systems/8. New data/tda_method_specs/vr_filtration_parameter_by_method.csv')
BASE_PANEL_PATH = Path('/Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin_with_euclidean/cmc15_benchmark_panel_with_euclidean.csv')
BASE_MANIFEST_PATH = Path('/Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin_with_euclidean/cmc15_indicator_manifest_with_euclidean.csv')
EVAL_WINDOWS_PATH = Path('/Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin/cmc15_evaluation_windows.csv')
TDA_DIR_MAIN = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv')
TDA_DIR_EU = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv_euclidean')

OUTPUT_ROOT = Path('/Users/jane/Documents/202511吾-Systems/8. New data/network_based_vs_tda')
SERIES_DIR = OUTPUT_ROOT / 'network_metric_series'
SERIES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SIM_FILES = sorted((SIM_ROOT / 'wasserstein').glob('*.npz')) + sorted((SIM_ROOT / 'dtw').glob('*.npz')) + sorted((SIM_ROOT / 'euclidean').glob('*.npz'))

PATTERN = re.compile(r'(wasserstein|dtw|euclidean)_similarity_m(\d+)_tau(\d+)')
NETWORK_FEATURES = [
    'degree_mean',
    'degree_std',
    'clustering_mean',
    'clustering_std',
    'pagerank_max',
    'pagerank_std',
]

print('Similarity files:', len(SIM_FILES))
print('Output root:', OUTPUT_ROOT)


Similarity files: 36
Output root: /Users/jane/Documents/202511吾-Systems/8. New data/network_based_vs_tda


In [2]:
events = pd.to_datetime([
    '2020-12-01','2021-01-02','2021-01-07','2021-01-29','2021-02-16','2021-03-13',
    '2021-04-10','2021-05-12','2021-05-17','2021-05-18','2021-05-19','2021-06-09',
    '2021-09-24','2021-10-15','2021-11-15','2022-04-27','2022-05-01','2022-05-11',
    '2022-05-12','2022-05-13','2022-07-20','2022-11-01','2023-03-01','2023-03-10',
    '2023-05-17','2023-06-16','2023-07-01','2023-10-01','2024-03-19','2024-04-20',
    '2025-01-20','2025-02-03','2025-02-21','2025-03-07','2025-05-20','2025-06-05','2025-06-17'
])

vr_params = pd.read_csv(VR_PARAM_PATH)
method_max_radius = dict(zip(vr_params['method'], vr_params['recommended_global_max_filtration_radius']))

base_panel = pd.read_csv(BASE_PANEL_PATH, parse_dates=['date'])
base_manifest = pd.read_csv(BASE_MANIFEST_PATH)
eval_windows = pd.read_csv(EVAL_WINDOWS_PATH)


def parse_similarity_name(path):
    match = PATTERN.search(path.stem)
    if not match:
        raise ValueError(f'Cannot parse method/window/lag from {path.name}')
    return match.group(1), int(match.group(2)), int(match.group(3))


def sanitize_distance_matrix(mat):
    out = np.nan_to_num(mat.astype(float), nan=0.0, posinf=0.0, neginf=0.0)
    out = (out + out.T) / 2.0
    np.fill_diagonal(out, 0.0)
    return out


def distance_to_affinity(dist_mat, max_radius):
    weights = 1.0 - (dist_mat / max_radius)
    weights = np.clip(weights, 0.0, 1.0)
    np.fill_diagonal(weights, 0.0)
    return weights


def compute_network_metrics(weights):
    n = weights.shape[0]
    degree = weights.sum(axis=1) / (n - 1)

    G = nx.from_numpy_array(weights)
    clustering = np.fromiter(nx.clustering(G, weight='weight').values(), dtype=float)
    pagerank = np.fromiter(nx.pagerank(G, alpha=0.85, weight='weight', max_iter=1000).values(), dtype=float)

    return {
        'degree_mean': float(degree.mean()),
        'degree_std': float(degree.std()),
        'clustering_mean': float(clustering.mean()),
        'clustering_std': float(clustering.std()),
        'pagerank_max': float(pagerank.max()),
        'pagerank_std': float(pagerank.std()),
    }


def evaluable_event_count(series_dates, events, start_offset, end_offset):
    series_start = pd.to_datetime(series_dates.min())
    series_end = pd.to_datetime(series_dates.max())
    count = 0
    for e in events:
        ws = e + pd.Timedelta(days=start_offset)
        we = e + pd.Timedelta(days=end_offset)
        if ws >= series_start and we <= series_end:
            count += 1
    return count


print('Global radius by method:', method_max_radius)
print('Base panel shape:', base_panel.shape)
print('Base manifest size:', len(base_manifest))


Global radius by method: {'dtw': 5.321885585784912, 'euclidean': 1.9430201053619385, 'wasserstein': 0.4271741211414337}
Base panel shape: (1867, 123)
Base manifest size: 115


## 1. Compute network-based indicators from the same distance matrices

We convert each distance matrix to a weighted network via

$$w_{ij} = \max\{0, 1 - d_{ij}/r_{\max}\},$$

where $r_{\max}$ is the method-specific global upper filtration radius already used in the TDA specification. We then compute weighted degree centrality, weighted clustering, and PageRank-based summaries.


In [3]:
network_manifest_rows = []
all_network_wide_parts = []
start_all = time.time()

for idx, sim_path in enumerate(SIM_FILES, start=1):
    method, window_size, lag = parse_similarity_name(sim_path)
    max_radius = method_max_radius[method]

    data = np.load(sim_path, allow_pickle=True)
    mats = data['similarity_matrices'].astype(float)
    dates = pd.to_datetime(data['dates']).astype(str)
    n_time = mats.shape[0]

    rows = []
    t0 = time.time()
    for t in range(n_time):
        dist_mat = sanitize_distance_matrix(mats[t])
        weights = distance_to_affinity(dist_mat, max_radius=max_radius)
        metrics = compute_network_metrics(weights)
        metrics.update({'date': dates[t], 'time_index': t})
        rows.append(metrics)

    metric_df = pd.DataFrame(rows)
    out_csv = SERIES_DIR / f'cmc15_network_{method}_m{window_size:02d}_tau{lag}.csv'
    metric_df.to_csv(out_csv, index=False)

    prefix = f'net_{method}_m{window_size:02d}_tau{lag}'
    part = metric_df[['date'] + NETWORK_FEATURES].rename(columns={c: f'{prefix}_{c}' for c in NETWORK_FEATURES})
    all_network_wide_parts.append(part)

    for feature in NETWORK_FEATURES:
        network_manifest_rows.append({
            'indicator': f'{prefix}_{feature}',
            'category': 'network',
            'distance_method': method,
            'embedding_window_size': window_size,
            'embedding_lag': lag,
            'feature': feature,
            'source_file': out_csv.name,
        })

    print(f'[{idx:02d}/{len(SIM_FILES)}] Saved {out_csv.name} ({time.time() - t0:.2f}s)')

network_manifest = pd.DataFrame(network_manifest_rows).sort_values(['distance_method', 'embedding_window_size', 'embedding_lag', 'feature']).reset_index(drop=True)
network_manifest.to_csv(OUTPUT_ROOT / 'cmc15_network_indicator_manifest.csv', index=False)

network_wide = all_network_wide_parts[0].copy()
for part in all_network_wide_parts[1:]:
    network_wide = network_wide.merge(part, on='date', how='outer')
network_wide['date'] = pd.to_datetime(network_wide['date'])
network_wide = network_wide.sort_values('date').reset_index(drop=True)
network_wide.to_csv(OUTPUT_ROOT / 'cmc15_network_wide_panel.csv', index=False)

combined_panel = base_panel.merge(network_wide, on='date', how='left')
combined_manifest = pd.concat([base_manifest, network_manifest], ignore_index=True)
combined_panel.to_csv(OUTPUT_ROOT / 'cmc15_benchmark_panel_with_network.csv', index=False)
combined_manifest.to_csv(OUTPUT_ROOT / 'cmc15_indicator_manifest_with_network.csv', index=False)

print(f'Total network-metric runtime: {time.time() - start_all:.2f}s')
print('Network wide shape:', network_wide.shape)
print('Combined panel shape:', combined_panel.shape)
print('Combined manifest size:', len(combined_manifest))
display(network_manifest.head(12))


[01/36] Saved cmc15_network_wasserstein_m07_tau1.csv (2.44s)


[02/36] Saved cmc15_network_wasserstein_m07_tau2.csv (2.53s)


[03/36] Saved cmc15_network_wasserstein_m07_tau3.csv (2.49s)


[04/36] Saved cmc15_network_wasserstein_m14_tau1.csv (2.51s)


[05/36] Saved cmc15_network_wasserstein_m14_tau2.csv (2.47s)


...

[34/36] Saved cmc15_network_euclidean_m28_tau1.csv (2.90s)


[35/36] Saved cmc15_network_euclidean_m28_tau2.csv (2.82s)


[36/36] Saved cmc15_network_euclidean_m28_tau3.csv (2.77s)


Total network-metric runtime: 93.57s
Network wide shape: (1867, 217)
Combined panel shape: (1867, 339)
Combined manifest size: 331


,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,source_file
0,net_dtw_m07_tau1_clustering_mean,network,dtw,7,1,clustering_mean,cmc15_network_dtw_m07_tau1.csv
1,net_dtw_m07_tau1_clustering_std,network,dtw,7,1,clustering_std,cmc15_network_dtw_m07_tau1.csv
2,net_dtw_m07_tau1_degree_mean,network,dtw,7,1,degree_mean,cmc15_network_dtw_m07_tau1.csv
3,net_dtw_m07_tau1_degree_std,network,dtw,7,1,degree_std,cmc15_network_dtw_m07_tau1.csv
4,net_dtw_m07_tau1_pagerank_max,network,dtw,7,1,pagerank_max,cmc15_network_dtw_m07_tau1.csv
5,net_dtw_m07_tau1_pagerank_std,network,dtw,7,1,pagerank_std,cmc15_network_dtw_m07_tau1.csv
6,net_dtw_m07_tau2_clustering_mean,network,dtw,7,2,clustering_mean,cmc15_network_dtw_m07_tau2.csv
7,net_dtw_m07_tau2_clustering_std,network,dtw,7,2,clustering_std,cmc15_network_dtw_m07_tau2.csv
8,net_dtw_m07_tau2_degree_mean,network,dtw,7,2,degree_mean,cmc15_network_dtw_m07_tau2.csv
9,net_dtw_m07_tau2_degree_std,network,dtw,7,2,degree_std,cmc15_network_dtw_m07_tau2.csv


## 2. Multi-window AUC comparison: TDA vs network indicators

We evaluate the network indicators using the same event labels and the same evaluation windows as in the revised benchmark design.


In [4]:
meta_map = combined_manifest.drop_duplicates('indicator').set_index('indicator').to_dict('index')
all_indicators = combined_manifest['indicator'].tolist()

auc_rows = []
for _, w in eval_windows.iterrows():
    window_label = w['window_label']
    start_offset = int(w['start_offset'])
    end_offset = int(w['end_offset'])
    label_col = f'event_label_{window_label}'
    y_full = combined_panel[label_col].to_numpy()

    for indicator in all_indicators:
        y_score = pd.to_numeric(combined_panel[indicator], errors='coerce').to_numpy()
        mask = np.isfinite(y_score)

        if mask.sum() == 0:
            continue

        y_true = y_full[mask]
        y_score_masked = y_score[mask]

        if np.unique(y_true).size < 2:
            continue

        fpr, tpr, _ = roc_curve(y_true, y_score_masked)
        roc_auc = auc(fpr, tpr)
        series_dates = combined_panel.loc[mask, 'date']
        meta = meta_map[indicator]

        auc_rows.append({
            'window_label': window_label,
            'start_offset': start_offset,
            'end_offset': end_offset,
            'indicator': indicator,
            'category': meta['category'],
            'distance_method': meta['distance_method'],
            'embedding_window_size': meta['embedding_window_size'],
            'embedding_lag': meta['embedding_lag'],
            'feature': meta['feature'],
            'auc': roc_auc,
            'n_obs': int(mask.sum()),
            'n_positive': int(y_true.sum()),
            'n_events_in_range': int(evaluable_event_count(series_dates, events, start_offset, end_offset)),
        })

auc_df = pd.DataFrame(auc_rows).sort_values(['window_label', 'auc'], ascending=[True, False]).reset_index(drop=True)
auc_df.to_csv(OUTPUT_ROOT / 'cmc15_auc_multiwindow_long_with_network.csv', index=False)

auc_mean = auc_df.groupby('indicator', as_index=False)['auc'].mean().rename(columns={'auc': 'mean_auc'})
auc_wide = auc_df.pivot(index='indicator', columns='window_label', values='auc').reset_index()
auc_summary = combined_manifest.drop_duplicates('indicator').merge(auc_mean, on='indicator', how='left').merge(auc_wide, on='indicator', how='left')
auc_summary = auc_summary.sort_values('mean_auc', ascending=False).reset_index(drop=True)
auc_summary.to_csv(OUTPUT_ROOT / 'cmc15_auc_indicator_summary_with_network.csv', index=False)

network_auc_summary = auc_summary.loc[auc_summary['category'] == 'network'].copy()
network_auc_summary.to_csv(OUTPUT_ROOT / 'cmc15_network_auc_indicator_summary.csv', index=False)

best_network = (
    auc_df.loc[auc_df['category'] == 'network']
    .sort_values(['window_label', 'auc'], ascending=[True, False])
    .drop_duplicates('window_label')
    .reset_index(drop=True)
)
best_network.to_csv(OUTPUT_ROOT / 'cmc15_network_auc_best_by_window.csv', index=False)

best_tda = (
    auc_df.loc[auc_df['category'] == 'tda']
    .sort_values(['window_label', 'auc'], ascending=[True, False])
    .drop_duplicates('window_label')
    .reset_index(drop=True)
)
best_tda.to_csv(OUTPUT_ROOT / 'cmc15_tda_auc_best_by_window_from_network_panel.csv', index=False)

tda_vs_network = best_tda.merge(
    best_network,
    on='window_label',
    suffixes=('_tda', '_network')
)
tda_vs_network['tda_minus_network_auc'] = tda_vs_network['auc_tda'] - tda_vs_network['auc_network']
tda_vs_network['winner'] = np.where(tda_vs_network['tda_minus_network_auc'] >= 0, 'TDA', 'Network')
tda_vs_network.to_csv(OUTPUT_ROOT / 'cmc15_tda_vs_network_best_by_window.csv', index=False)

best_network_by_distance = (
    auc_df.loc[auc_df['category'] == 'network']
    .sort_values(['window_label', 'distance_method', 'auc'], ascending=[True, True, False])
    .drop_duplicates(['window_label', 'distance_method'])
    .reset_index(drop=True)
)
best_network_by_distance.to_csv(OUTPUT_ROOT / 'cmc15_network_auc_best_by_distance_window.csv', index=False)

best_tda_by_distance = (
    auc_df.loc[auc_df['category'] == 'tda']
    .sort_values(['window_label', 'distance_method', 'auc'], ascending=[True, True, False])
    .drop_duplicates(['window_label', 'distance_method'])
    .reset_index(drop=True)
)
best_tda_by_distance.to_csv(OUTPUT_ROOT / 'cmc15_tda_auc_best_by_distance_window_from_network_panel.csv', index=False)

tda_vs_network_by_distance = best_tda_by_distance.merge(
    best_network_by_distance,
    on=['window_label', 'distance_method'],
    suffixes=('_tda', '_network')
)
tda_vs_network_by_distance['tda_minus_network_auc'] = tda_vs_network_by_distance['auc_tda'] - tda_vs_network_by_distance['auc_network']
tda_vs_network_by_distance['winner'] = np.where(tda_vs_network_by_distance['tda_minus_network_auc'] >= 0, 'TDA', 'Network')
tda_vs_network_by_distance.to_csv(OUTPUT_ROOT / 'cmc15_tda_vs_network_best_by_distance_window.csv', index=False)

display(tda_vs_network)
display(tda_vs_network_by_distance.head(15))


,window_label,start_offset_tda,end_offset_tda,indicator_tda,category_tda,distance_method_tda,embedding_window_size_tda,embedding_lag_tda,feature_tda,auc_tda,...,distance_method_network,embedding_window_size_network,embedding_lag_network,feature_network,auc_network,n_obs_network,n_positive_network,n_events_in_range_network,tda_minus_network_auc,winner
0,"[-14,0]",-14,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.636218,...,wasserstein,28.0,3.0,pagerank_max,0.618942,1792,452,37,0.017275,TDA
1,"[-21,0]",-21,0,tda_euclidean_m28_tau3_betti0,tda,euclidean,28.0,3.0,betti0,0.664299,...,wasserstein,28.0,3.0,pagerank_max,0.646626,1792,604,37,0.017673,TDA
2,"[-7,0]",-7,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.649221,...,wasserstein,7.0,1.0,pagerank_max,0.634009,1867,258,37,0.015212,TDA
3,"[0,+21]",0,21,tda_wasserstein_m21_tau1_betti0,tda,wasserstein,21.0,1.0,betti0,0.688595,...,wasserstein,28.0,1.0,pagerank_max,0.677268,1846,604,37,0.011326,TDA
4,"[0,+7]",0,7,tda_dtw_m14_tau1_betti0,tda,dtw,14.0,1.0,betti0,0.649443,...,euclidean,7.0,2.0,pagerank_max,0.635526,1861,258,37,0.013917,TDA


,window_label,start_offset_tda,end_offset_tda,indicator_tda,category_tda,distance_method,embedding_window_size_tda,embedding_lag_tda,feature_tda,auc_tda,...,category_network,embedding_window_size_network,embedding_lag_network,feature_network,auc_network,n_obs_network,n_positive_network,n_events_in_range_network,tda_minus_network_auc,winner
0,"[-14,0]",-14,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.636218,...,network,7.0,1.0,pagerank_max,0.604586,1867,452,37,0.031632,TDA
1,"[-14,0]",-14,0,tda_euclidean_m28_tau3_betti0,tda,euclidean,28.0,3.0,betti0,0.633098,...,network,28.0,3.0,pagerank_std,0.610532,1792,452,37,0.022566,TDA
2,"[-14,0]",-14,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.628166,...,network,28.0,3.0,pagerank_max,0.618942,1792,452,37,0.009224,TDA
3,"[-21,0]",-21,0,tda_dtw_m28_tau3_betti0,tda,dtw,28.0,3.0,betti0,0.662981,...,network,28.0,3.0,pagerank_max,0.637182,1792,604,37,0.025799,TDA
4,"[-21,0]",-21,0,tda_euclidean_m28_tau3_betti0,tda,euclidean,28.0,3.0,betti0,0.664299,...,network,28.0,3.0,pagerank_max,0.640032,1792,604,37,0.024267,TDA
5,"[-21,0]",-21,0,tda_wasserstein_m28_tau3_betti0,tda,wasserstein,28.0,3.0,betti0,0.649236,...,network,28.0,3.0,pagerank_max,0.646626,1792,604,37,0.002610,TDA
6,"[-7,0]",-7,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.646456,...,network,7.0,1.0,pagerank_max,0.626982,1867,258,37,0.019474,TDA
7,"[-7,0]",-7,0,tda_euclidean_m07_tau1_betti0,tda,euclidean,7.0,1.0,betti0,0.640397,...,network,7.0,1.0,pagerank_max,0.632778,1867,258,37,0.007619,TDA
8,"[-7,0]",-7,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.649221,...,network,7.0,1.0,pagerank_max,0.634009,1867,258,37,0.015212,TDA
9,"[0,+21]",0,21,tda_dtw_m28_tau1_betti0,tda,dtw,28.0,1.0,betti0,0.687707,...,network,7.0,2.0,pagerank_max,0.660550,1861,604,37,0.027157,TDA


## 3. Relationship analysis between TDA and network indicators

We compute Spearman correlations between TDA features and network features for each matching `(distance method, m, tau)` specification. We also compare the best TDA feature and the best network feature within each matching specification based on mean AUC across the five evaluation windows.


In [5]:
tda_paths = sorted(TDA_DIR_MAIN.glob('cmc15_tda_*.csv')) + sorted(TDA_DIR_EU.glob('cmc15_tda_euclidean_m*_tau*.csv'))
tda_lookup = {}

for path in tda_paths:
    name = path.name
    if 'manifest' in name:
        continue
    if name.startswith('cmc15_tda_euclidean_'):
        m = re.search(r'm(\d+)_tau(\d+)', name)
        key = ('euclidean', int(m.group(1)), int(m.group(2)))
    else:
        m = re.search(r'cmc15_tda_(wasserstein|dtw)_m(\d+)_tau(\d+)', name)
        key = (m.group(1), int(m.group(2)), int(m.group(3)))
    tda_lookup[key] = path

network_file_lookup = {}
for path in sorted(SERIES_DIR.glob('cmc15_network_*.csv')):
    m = re.search(r'cmc15_network_(wasserstein|dtw|euclidean)_m(\d+)_tau(\d+)', path.name)
    network_file_lookup[(m.group(1), int(m.group(2)), int(m.group(3)))] = path

corr_rows = []
pairwise_auc_rows = []

tda_summary = auc_summary.loc[auc_summary['category'] == 'tda'].copy()
network_summary = auc_summary.loc[auc_summary['category'] == 'network'].copy()

for key in sorted(network_file_lookup.keys()):
    method, window_size, lag = key
    tda_path = tda_lookup[key]
    net_path = network_file_lookup[key]

    tda_df = pd.read_csv(tda_path, parse_dates=['date'])
    net_df = pd.read_csv(net_path, parse_dates=['date'])
    merged = tda_df[['date', 'betti0', 'betti1', 'persistent_entropy']].merge(net_df[['date'] + NETWORK_FEATURES], on='date', how='inner')

    for tda_feature in ['betti0', 'betti1', 'persistent_entropy']:
        for network_feature in NETWORK_FEATURES:
            rho, pval = spearmanr(merged[tda_feature], merged[network_feature], nan_policy='omit')
            corr_rows.append({
                'distance_method': method,
                'embedding_window_size': window_size,
                'embedding_lag': lag,
                'tda_feature': tda_feature,
                'network_feature': network_feature,
                'spearman_rho': rho,
                'spearman_abs_rho': abs(rho) if pd.notna(rho) else np.nan,
                'p_value': pval,
                'n_obs': len(merged),
            })

    tda_sub = tda_summary[
        (tda_summary['distance_method'] == method)
        & (tda_summary['embedding_window_size'] == window_size)
        & (tda_summary['embedding_lag'] == lag)
    ].sort_values('mean_auc', ascending=False)

    net_sub = network_summary[
        (network_summary['distance_method'] == method)
        & (network_summary['embedding_window_size'] == window_size)
        & (network_summary['embedding_lag'] == lag)
    ].sort_values('mean_auc', ascending=False)

    best_tda_row = tda_sub.iloc[0]
    best_network_row = net_sub.iloc[0]
    pairwise_auc_rows.append({
        'distance_method': method,
        'embedding_window_size': window_size,
        'embedding_lag': lag,
        'best_tda_indicator': best_tda_row['indicator'],
        'best_tda_feature': best_tda_row['feature'],
        'best_tda_mean_auc': best_tda_row['mean_auc'],
        'best_network_indicator': best_network_row['indicator'],
        'best_network_feature': best_network_row['feature'],
        'best_network_mean_auc': best_network_row['mean_auc'],
        'tda_minus_network_mean_auc': best_tda_row['mean_auc'] - best_network_row['mean_auc'],
        'winner': 'TDA' if best_tda_row['mean_auc'] >= best_network_row['mean_auc'] else 'Network',
    })

corr_df = pd.DataFrame(corr_rows).sort_values(['distance_method', 'embedding_window_size', 'embedding_lag', 'tda_feature', 'network_feature']).reset_index(drop=True)
corr_df.to_csv(OUTPUT_ROOT / 'cmc15_tda_network_correlation_long.csv', index=False)

corr_summary = (
    corr_df.groupby(['tda_feature', 'network_feature'], as_index=False)
    .agg(
        mean_spearman_rho=('spearman_rho', 'mean'),
        median_spearman_rho=('spearman_rho', 'median'),
        mean_abs_spearman_rho=('spearman_abs_rho', 'mean'),
        max_abs_spearman_rho=('spearman_abs_rho', 'max'),
        n_pairs=('spearman_rho', 'count'),
    )
    .sort_values('mean_abs_spearman_rho', ascending=False)
    .reset_index(drop=True)
)
corr_summary.to_csv(OUTPUT_ROOT / 'cmc15_tda_network_correlation_summary.csv', index=False)

pairwise_auc_df = pd.DataFrame(pairwise_auc_rows).sort_values(['distance_method', 'embedding_window_size', 'embedding_lag']).reset_index(drop=True)
pairwise_auc_df.to_csv(OUTPUT_ROOT / 'cmc15_tda_vs_network_pairwise_mean_auc_summary.csv', index=False)

pairwise_group_summary = (
    pairwise_auc_df.groupby('distance_method', as_index=False)
    .agg(
        n_specifications=('distance_method', 'count'),
        tda_win_count=('winner', lambda s: int((s == 'TDA').sum())),
        network_win_count=('winner', lambda s: int((s == 'Network').sum())),
        mean_tda_minus_network_auc=('tda_minus_network_mean_auc', 'mean'),
        median_tda_minus_network_auc=('tda_minus_network_mean_auc', 'median'),
        max_tda_minus_network_auc=('tda_minus_network_mean_auc', 'max'),
        min_tda_minus_network_auc=('tda_minus_network_mean_auc', 'min'),
    )
)
pairwise_group_summary.to_csv(OUTPUT_ROOT / 'cmc15_tda_vs_network_pairwise_group_summary.csv', index=False)

display(corr_summary.head(15))
display(pairwise_auc_df.head(12))
display(pairwise_group_summary)


,tda_feature,network_feature,mean_spearman_rho,median_spearman_rho,mean_abs_spearman_rho,max_abs_spearman_rho,n_pairs
0,betti0,degree_mean,-0.974817,-0.973637,0.974817,0.987910,36
1,betti0,clustering_mean,-0.972402,-0.971423,0.972402,0.986842,36
2,betti0,pagerank_max,0.925281,0.927530,0.925281,0.960809,36
3,betti0,pagerank_std,0.852571,0.842466,0.852571,0.939616,36
4,betti0,clustering_std,0.842122,0.831384,0.842122,0.936742,36
5,betti0,degree_std,0.841901,0.831726,0.841901,0.936206,36
6,persistent_entropy,pagerank_max,-0.161859,-0.172416,0.161859,0.239255,36
7,persistent_entropy,degree_std,-0.127771,-0.139908,0.130168,0.225679,36
8,persistent_entropy,pagerank_std,-0.127070,-0.139506,0.129384,0.223436,36
9,persistent_entropy,clustering_std,-0.126960,-0.138573,0.129364,0.225233,36


,distance_method,embedding_window_size,embedding_lag,best_tda_indicator,best_tda_feature,best_tda_mean_auc,best_network_indicator,best_network_feature,best_network_mean_auc,tda_minus_network_mean_auc,winner
0,dtw,7,1,tda_dtw_m07_tau1_betti0,betti0,0.641418,net_dtw_m07_tau1_pagerank_max,pagerank_max,0.618600,0.022817,TDA
1,dtw,7,2,tda_dtw_m07_tau2_betti0,betti0,0.635214,net_dtw_m07_tau2_pagerank_max,pagerank_max,0.617753,0.017461,TDA
2,dtw,7,3,tda_dtw_m07_tau3_betti0,betti0,0.630672,net_dtw_m07_tau3_pagerank_max,pagerank_max,0.612946,0.017726,TDA
3,dtw,14,1,tda_dtw_m14_tau1_betti0,betti0,0.644919,net_dtw_m14_tau1_pagerank_max,pagerank_max,0.613116,0.031802,TDA
4,dtw,14,2,tda_dtw_m14_tau2_betti0,betti0,0.628023,net_dtw_m14_tau2_pagerank_max,pagerank_max,0.595694,0.032329,TDA
5,dtw,14,3,tda_dtw_m14_tau3_betti0,betti0,0.623719,net_dtw_m14_tau3_pagerank_max,pagerank_max,0.593354,0.030365,TDA
6,dtw,21,1,tda_dtw_m21_tau1_betti0,betti0,0.635389,net_dtw_m21_tau1_pagerank_max,pagerank_max,0.600013,0.035376,TDA
7,dtw,21,2,tda_dtw_m21_tau2_betti0,betti0,0.625887,net_dtw_m21_tau2_pagerank_max,pagerank_max,0.595229,0.030658,TDA
8,dtw,21,3,tda_dtw_m21_tau3_betti0,betti0,0.622075,net_dtw_m21_tau3_pagerank_std,pagerank_std,0.590341,0.031734,TDA
9,dtw,28,1,tda_dtw_m28_tau1_betti0,betti0,0.633964,net_dtw_m28_tau1_pagerank_max,pagerank_max,0.593108,0.040856,TDA


,distance_method,n_specifications,tda_win_count,network_win_count,mean_tda_minus_network_auc,median_tda_minus_network_auc,max_tda_minus_network_auc,min_tda_minus_network_auc
0,dtw,12,12,0,0.028593,0.030512,0.040856,0.017461
1,euclidean,12,12,0,0.021030,0.023161,0.030176,0.001674
2,wasserstein,12,12,0,0.015089,0.015526,0.021085,0.006166


In [6]:
manifest = pd.DataFrame({
    'file_name': [
        'cmc15_network_indicator_manifest.csv',
        'cmc15_network_wide_panel.csv',
        'cmc15_benchmark_panel_with_network.csv',
        'cmc15_indicator_manifest_with_network.csv',
        'cmc15_auc_multiwindow_long_with_network.csv',
        'cmc15_auc_indicator_summary_with_network.csv',
        'cmc15_network_auc_indicator_summary.csv',
        'cmc15_network_auc_best_by_window.csv',
        'cmc15_tda_auc_best_by_window_from_network_panel.csv',
        'cmc15_tda_vs_network_best_by_window.csv',
        'cmc15_network_auc_best_by_distance_window.csv',
        'cmc15_tda_auc_best_by_distance_window_from_network_panel.csv',
        'cmc15_tda_vs_network_best_by_distance_window.csv',
        'cmc15_tda_network_correlation_long.csv',
        'cmc15_tda_network_correlation_summary.csv',
        'cmc15_tda_vs_network_pairwise_mean_auc_summary.csv',
        'cmc15_tda_vs_network_pairwise_group_summary.csv',
    ]
})
manifest['path'] = manifest['file_name'].map(lambda x: str(OUTPUT_ROOT / x))
manifest.to_csv(OUTPUT_ROOT / 'network_based_vs_tda_manifest.csv', index=False)

print('Saved files:')
for name in manifest['file_name']:
    print('-', name)
display(manifest)


Saved files:
- cmc15_network_indicator_manifest.csv
- cmc15_network_wide_panel.csv
- cmc15_benchmark_panel_with_network.csv
- cmc15_indicator_manifest_with_network.csv
- cmc15_auc_multiwindow_long_with_network.csv
- cmc15_auc_indicator_summary_with_network.csv
- cmc15_network_auc_indicator_summary.csv
- cmc15_network_auc_best_by_window.csv
- cmc15_tda_auc_best_by_window_from_network_panel.csv
- cmc15_tda_vs_network_best_by_window.csv
- cmc15_network_auc_best_by_distance_window.csv
- cmc15_tda_auc_best_by_distance_window_from_network_panel.csv
- cmc15_tda_vs_network_best_by_distance_window.csv
- cmc15_tda_network_correlation_long.csv
- cmc15_tda_network_correlation_summary.csv
- cmc15_tda_vs_network_pairwise_mean_auc_summary.csv
- cmc15_tda_vs_network_pairwise_group_summary.csv


,file_name,path
0,cmc15_network_indicator_manifest.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
1,cmc15_network_wide_panel.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
2,cmc15_benchmark_panel_with_network.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
3,cmc15_indicator_manifest_with_network.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
4,cmc15_auc_multiwindow_long_with_network.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
5,cmc15_auc_indicator_summary_with_network.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
6,cmc15_network_auc_indicator_summary.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
7,cmc15_network_auc_best_by_window.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
8,cmc15_tda_auc_best_by_window_from_network_pane...,/Users/jane/Documents/202511吾-Systems/8. New d...
9,cmc15_tda_vs_network_best_by_window.csv,/Users/jane/Documents/202511吾-Systems/8. New d...
